In [1]:
from astropy.utils import iers
iers.conf.auto_download = False

import os
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"   # must be set BEFORE importing torch

import io, re, time
import pathlib
import requests
import numpy as np
import pandas as pd
import torch
from collections import Counter

In [2]:
import pathlib
import numpy as np
import pandas as pd

here = pathlib.Path.cwd()
root = here.parent if here.name.lower() == "notebooks" else here
assert root.name == "Sunspot detection with ML", f"Wrong root: {root}"

DATA      = root / "Data"
RAW_HMI   = DATA / "Raw" / "HMI"
REF_DEB   = DATA / "Reference" / "Debrecen"
REF_NOAA  = DATA / "Reference" / "NOAA"
PROCESSED = DATA / "Processed"

for d in [RAW_HMI, REF_DEB, REF_NOAA, PROCESSED, root / "Src"]:
    d.mkdir(parents=True, exist_ok=True)

print("Project root:", root)

Project root: /Users/vinuradesilva/Documents/Projects/Sunspot detection with ML


In [3]:
print("torch:", torch.__version__)
print("MPS built    :", torch.backends.mps.is_built())
print("MPS available:", torch.backends.mps.is_available())

device = torch.device("mps") if torch.backends.mps.is_available() else torch.device("cpu")
x = torch.randn(2000, 2000, device=device)
y = x @ x
print("Computed on:", y.device, "| dtype:", y.dtype)

torch: 2.14.1
MPS built    : True
MPS available: True
Computed on: mps:0 | dtype: torch.float32


In [4]:
for name in ["HMIDD2014.txt", "fdHMIDD2014.txt"]:
    f = REF_DEB / name
    with open(f) as fh:
        lines = fh.readlines()
    print(f"\n=== {name}: {f.stat().st_size/1e6:.1f} MB, {len(lines)} lines")
    print("Row types:", Counter(l[0] for l in lines if l.strip()))
    for l in lines[:8]:
        print(repr(l))


=== HMIDD2014.txt: 283.0 MB, 2111146 lines
Row types: Counter({'s': 1961470, 'g': 141014, 'h': 8662})
'h 2014 01 01 00 00 24.20 SHMI          19   333.46  1302.79   542.98  1546.36 2456658.50028009    0.07  -3.00 189.47  999999  999999  999999\n'
'g 2014 01 01 00 00 24.20   11934       11    12.57    68.29    48.07   244.41   -15.35  271.70   82.23 254.87 0.9888    44.5    57.4\n'
's 2014 01 01 00 00 24.20   11934        1     6.03    29.48    28.56   146.87   -14.07  273.90   84.43 256.16 0.9948  -130.0    39.5\n'
's 2014 01 01 00 00 24.20   11934        2     4.67    13.59    13.89    40.44   -16.30  270.31   80.84 253.96 0.9867   423.9   276.4\n'
's 2014 01 01 00 00 24.20   11934        3     0.00     1.61     0.00     5.36   -16.81  271.32   81.85 253.45 0.9893  -111.0  -168.7\n'
's 2014 01 01 00 00 24.20   11934        4     0.00     1.10     0.00     3.27   -16.76  270.38   80.91 253.50 0.9869   363.0   339.9\n'
's 2014 01 01 00 00 24.20   11934        5     1.87     5.44     5.

In [5]:
SRC = REF_DEB / "HMIDD2014.txt"
print("Reading:", SRC.name)          # the printed name must say HMIDD2014.txt

cols = ["code","year","month","day","hour","minute","second","grp","n",
        "c10","c11","c12","c13","c14","c15","c16","c17","c18","c19","c20"]

keep, bad = [], 0
with open(SRC) as fh:
    for line in fh:
        if line[0] in "hg":
            if len(line.split()) != 20:
                bad += 1
            keep.append(line)
print(f"kept {len(keep)} h/g rows | malformed: {bad}")

# Safety check: fail loudly if this is not the grouped file
assert len(keep) > 100_000, f"Only {len(keep)} h/g rows: wrong file?"

df = pd.read_csv(io.StringIO("".join(keep)), sep=r"\s+", header=None,
                 names=cols, dtype={"code": str, "grp": str})
df = df.replace(999999, np.nan)
df["time"] = (pd.to_datetime(df[["year","month","day","hour","minute"]])
              + pd.to_timedelta(df["second"], unit="s"))

h = df[df["code"] == "h"].rename(columns={
    "n":"n_groups","c10":"proj_U","c11":"proj_UP","c12":"corr_U",
    "c13":"corr_UP","c14":"jd","c15":"P","c16":"B0","c17":"L0"})
h = h[["time","n_groups","proj_U","proj_UP","corr_U","corr_UP","jd","P","B0","L0"]]

g = df[df["code"] == "g"].rename(columns={
    "grp":"noaa","n":"n_spots","c10":"proj_U","c11":"proj_UP","c12":"corr_U",
    "c13":"corr_UP","c14":"lat","c15":"lon","c16":"lcm","c18":"r"})
g = g[["time","noaa","n_spots","proj_U","proj_UP","corr_U","corr_UP","lat","lon","lcm","r"]]

print(f"h rows: {len(h)} | g rows: {len(g)} | distinct group labels: {g['noaa'].nunique()}")
print("first few labels:", list(g["noaa"].unique()[:5]))

per_day = h.groupby(h["time"].dt.date).size()
print(f"days with data: {len(per_day)} | hourly rows per day (min/median/max): "
      f"{per_day.min()} / {per_day.median()} / {per_day.max()}")

day = g[(g["noaa"] == "12192") & (g["time"].dt.date == pd.Timestamp("2014-10-23").date())]
print(f"\n{len(day)} hourly rows for AR 12192 on 2014-10-23")
print(day.iloc[::4].to_string(index=False, float_format=lambda x: f"{x:.2f}"))

Reading: HMIDD2014.txt
kept 149676 h/g rows | malformed: 0
h rows: 8662 | g rows: 141014 | distinct group labels: 3502
first few labels: ['11934', '11934m', '11935m', '11935n', '11935p']
days with data: 365 | hourly rows per day (min/median/max): 16 / 24.0 / 25

24 hourly rows for AR 12192 on 2014-10-23
                   time  noaa  n_spots  proj_U  proj_UP  corr_U  corr_UP    lat    lon   lcm    r
2014-10-23 00:00:30.100 12192      398 1250.49  6527.96  665.03  3474.60 -13.33 244.90 -9.59 0.36
2014-10-23 04:00:30.100 12192      409 1155.07  5883.82  610.89  3112.73 -13.30 245.33 -6.96 0.35
2014-10-23 08:00:30.100 12192      406 1235.14  6639.33  650.58  3497.89 -13.37 245.20 -4.91 0.34
2014-10-23 12:00:30.000 12192      390 1203.23  6387.69  632.14  3355.44 -13.40 245.33 -2.59 0.33
2014-10-23 16:00:29.900 12192      378 1226.06  6564.85  643.38  3449.63 -13.59 245.41 -0.32 0.33
2014-10-23 20:00:29.800 12192      412 1296.39  6353.85  680.11  3339.45 -13.48 245.37  1.85 0.33


In [6]:
# 1. Test corrected ≈ proj_UP / (2 cos θ) on many groups
deb_test = g[(g["r"] < 0.6) & (g["proj_UP"] > 50)].copy()
cos_t = np.sqrt(1 - deb_test["r"]**2)
deb_test["pred"] = deb_test["proj_UP"] / (2 * cos_t)
deb_test["ratio"] = deb_test["corr_UP"] / deb_test["pred"]
print(f"{len(deb_test)} groups tested (r < 0.6, projected U+P > 50)")
print(deb_test["ratio"].describe().round(3).to_string())

# 2. Save the parsed h and g tables
h.to_csv(PROCESSED / "debrecen_h_2014.csv", index=False)
g.to_csv(PROCESSED / "debrecen_g_2014.csv", index=False)
print("\nsaved:", [p.name for p in PROCESSED.iterdir()])

19764 groups tested (r < 0.6, projected U+P > 50)
count    19764.000
mean         0.991
std          0.002
min          0.982
25%          0.990
50%          0.991
75%          0.992
max          1.029

saved: ['debrecen_h_2014.csv', 'debrecen_g_2014.csv', 'noaa_srs_test_case.csv']


In [7]:
def read_srs(path):
    with open(path) as fh:
        lines = fh.readlines()
    issued = next(l for l in lines if l.startswith(":Issued:"))
    issued = pd.to_datetime(issued.split(":Issued:")[1].replace("UTC", "").strip(),
                            format="%Y %b %d %H%M")
    valid = issued.normalize()          # report is valid at 24:00Z of the day before = 00:00 of issue date

    rows, short, on = [], 0, False
    for l in lines:
        if l.startswith("I.  Regions with Sunspots"):
            on = True; continue
        if on and (l.startswith("IA.") or l.startswith("II.")):
            break
        if on and re.match(r"^\d{4}\s", l):
            p = l.split()
            if len(p) != 8:
                short += 1
            rows.append((p + [None] * 8)[:8])
    d = pd.DataFrame(rows, columns=["nmbr","location","lo","area","z","ll","nn","mag"])
    for c in ["area", "ll", "nn"]:
        d[c] = pd.to_numeric(d[c])
    d["noaa"] = (d["nmbr"].astype(int) + 10000).astype(str)   # 2192 -> "12192"
    d["valid"] = valid
    print(f"{path.name}: {len(d)} regions, valid {valid}, rows without 8 fields: {short}")
    return d

srs = read_srs(REF_NOAA / "20141024SRS.txt")

# Debrecen group rows within 30 min of the valid time, nearest one per group
v = srs["valid"].iloc[0]
near = g[(g["time"] - v).abs() <= pd.Timedelta("30min")].copy()
near["dt"] = (near["time"] - v).abs()
near = near.sort_values("dt").drop_duplicates("noaa")

cmp = srs.merge(near[["noaa", "corr_UP", "n_spots", "lat", "lcm"]], on="noaa", how="left")
cmp["area_ratio"] = cmp["area"] / cmp["corr_UP"]
print(cmp[["noaa", "location", "z", "area", "corr_UP", "area_ratio", "nn", "n_spots"]]
      .to_string(index=False, float_format=lambda x: f"{x:.2f}"))

20141024SRS.txt: 5 regions, valid 2014-10-24 00:00:00, rows without 8 fields: 0
 noaa location   z  area  corr_UP  area_ratio  nn  n_spots
12187   S09W81 Hsx   110   283.52        0.39   1        1
12192   S14W06 Fkc  2740  3688.40        0.74  58      407
12193   N06W45 Dao   100   177.64        0.56   8       25
12194   S12E32 Cso    70   155.69        0.45   2       18
12195   N07E59 Dso    70   162.57        0.43   7       20


In [8]:
BASE = "https://www.ngdc.noaa.gov/stp/space-weather/swpc-products/daily_reports/solar_region_summaries"

def srs_url(date):
    year = date.strftime("%Y")
    month = date.strftime("%m")
    stamp = date.strftime("%Y%m%d")
    return f"{BASE}/{year}/{month}/{stamp}SRS.txt"

# test = srs_url(pd.Timestamp("2014-10-24"))
# print(test)
# assert test.endswith("/2014/10/20141024SRS.txt"), "URL is wrong"

test = srs_url(pd.Timestamp("2014-02-03"))
print(test)
assert test.endswith("/2014/02/20140203SRS.txt"), "URL is wrong"

https://www.ngdc.noaa.gov/stp/space-weather/swpc-products/daily_reports/solar_region_summaries/2014/02/20140203SRS.txt


In [9]:
eval_periods = {
    "validation" : [("2013-01-01", "2013-03-31")],
    "test" : [("2014-02-01", "2014-02-28"), ("2014-06-01", "2014-06-30")],
    "case_study" : [("2014-10-01", "2014-10-31")],
}

dates = {}
for split, ranges in eval_periods.items():
    dates[split] = []
    for start, end in ranges:
        dates[split] += list(pd.date_range(start, end, freq="D"))

for split, days in dates.items():
    print(split, len(days), days[0].date(), "to", days[-1].date())


validation 90 2013-01-01 to 2013-03-31
test 58 2014-02-01 to 2014-06-30
case_study 31 2014-10-01 to 2014-10-31


In [10]:
def download_srs(days, pause=60):
    saved, skipped, failed = [], [], []
    for day in days:
        url  = srs_url(day)
        path = REF_NOAA / url.split("/")[-1]      # file name = last part of the address

        if path.exists():                         # already downloaded?
            skipped.append(day.date())
            continue

        try:
            r = requests.get(url, timeout=30)
            got_file = (r.status_code == 200) and ("Solar Region Summary" in r.text)
        except requests.RequestException as e:
            ok = False
            print("request problem:", day.date(), e)

        if got_file:
            path.write_text(r.text)
            saved.append(day.date())
        else:
            failed.append(day.date())

        print(day.date(), "ok" if ok else "FAILED")
        time.sleep(pause)

    print(f"saved {len(saved)} | skipped {len(skipped)} | failed {len(failed)}")
    return failed

In [11]:
batch1 = dates["test"] + dates["case_study"]
print(len(batch1), "days in batch 1")

failed = download_srs(batch1)
print("failed:", failed)

89 days in batch 1
saved 0 | skipped 89 | failed 0
failed: []


In [12]:
bad = []
files = sorted(REF_NOAA.glob("*SRS.txt"))

for f in files:
    stamp_from_name = f.name[:8]
    lines = f.read_text().splitlines()
    issued_line = next(l for l in lines if l.startswith(":Issued:"))
    issued = pd.to_datetime(issued_line.split(":Issued:")[1].replace("UTC", "").strip(),format="%Y %b %d %H%M")
    stamp_from_file = issued.strftime("%Y%m%d")

    if stamp_from_name != stamp_from_file:
        bad.append(f.name)

print(f"{len(files)} files checked | mismatches: {len(bad)} {bad}")

89 files checked | mismatches: 0 []


In [13]:
tables = []

for f in sorted(REF_NOAA.glob("*SRS.txt")):
    tables.append(read_srs(f))

noaa = pd.concat(tables, ignore_index=True)
print(noaa.shape) 

20140201SRS.txt: 2 regions, valid 2014-02-01 00:00:00, rows without 8 fields: 0
20140202SRS.txt: 3 regions, valid 2014-02-02 00:00:00, rows without 8 fields: 0
20140203SRS.txt: 3 regions, valid 2014-02-03 00:00:00, rows without 8 fields: 0
20140204SRS.txt: 4 regions, valid 2014-02-04 00:00:00, rows without 8 fields: 0
20140205SRS.txt: 5 regions, valid 2014-02-05 00:00:00, rows without 8 fields: 0
20140206SRS.txt: 8 regions, valid 2014-02-06 00:00:00, rows without 8 fields: 0
20140207SRS.txt: 7 regions, valid 2014-02-07 00:00:00, rows without 8 fields: 0
20140208SRS.txt: 7 regions, valid 2014-02-08 00:00:00, rows without 8 fields: 0
20140209SRS.txt: 7 regions, valid 2014-02-09 00:00:00, rows without 8 fields: 0
20140210SRS.txt: 7 regions, valid 2014-02-10 00:00:00, rows without 8 fields: 0
20140211SRS.txt: 6 regions, valid 2014-02-11 00:00:00, rows without 8 fields: 0
20140212SRS.txt: 7 regions, valid 2014-02-12 00:00:00, rows without 8 fields: 0
20140213SRS.txt: 8 regions, valid 2014-0

In [14]:
assert noaa["valid"].nunique() == 89
row = noaa[(noaa["noaa"] == "12192") & (noaa["valid"] == pd.Timestamp("2014-10-24"))]
assert row["area"].iloc[0] == 2740
print("Both checks passed")

Both checks passed


In [15]:
noaa.to_csv(PROCESSED / "noaa_srs_test_case.csv", index=False)

In [16]:
noaa_s = noaa.sort_values("valid").copy()
g_s    = g.sort_values("time").copy()

noaa_s["valid"] = noaa_s["valid"].astype("datetime64[ns]")
g_s["time"]     = g_s["time"].astype("datetime64[ns]")

m = pd.merge_asof(
    noaa_s,
    g_s[["time", "noaa", "corr_UP", "n_spots", "lat", "lcm", "r"]],
    left_on="valid",
    right_on="time",
    by="noaa",
    tolerance=pd.Timedelta("30min"),
    direction="nearest",
)

m["ratio"] = m["area"] / m["corr_UP"]
print(len(m), "NOAA rows |", m["corr_UP"].notna().sum(), "matched to Debrecen")

chk = m[(m["noaa"] == "12192") & (m["valid"] == pd.Timestamp("2014-10-24"))]
assert abs(chk["ratio"].iloc[0] - 0.743) < 0.01, "does not match the earlier result"
print("matches the earlier 24 October comparison")

544 NOAA rows | 520 matched to Debrecen
matches the earlier 24 October comparison


In [17]:
un = m[m["corr_UP"].isna()]
print(len(un), "unmatched")
print(un[["noaa", "valid", "location", "area", "z", "nn"]].to_string(index=False))

print("\nNOAA area, unmatched:")
print(un["area"].describe())
print("\nNOAA area, matched:")
print(m.loc[m["corr_UP"].notna(), "area"].describe())

24 unmatched
 noaa      valid location  area   z  nn
11969 2014-02-06   S10W87    10 Bxo   2
11979 2014-02-13   N14W40    10 Bxo   2
11971 2014-02-15   S12W85    30 Hsx   1
11976 2014-02-22   S13W94    60 Hax   1
11985 2014-02-23   N07W86    10 Bxo   2
11977 2014-02-23   S08W85    10 Axx   1
11988 2014-02-28   S10W37    60 Dao  10
12073 2014-06-02   S12W80     0 Axx   1
12078 2014-06-07   S20W57     0 Axx   1
12083 2014-06-08   S12W91    10 Axx   1
12079 2014-06-14   N12W94    90 Hsx   1
12080 2014-06-15   S11W91   350 Dki   8
12082 2014-06-16   N15W93    60 Cao   2
12095 2014-06-20   N05W84    20 Cro   4
12090 2014-06-24   N25W90    30 Hsx   1
12093 2014-06-27   S03W77    40 Cao   3
12094 2014-06-27   N11W68    10 Axx   1
12173 2014-10-03   S13W85    50 Hax   1
12172 2014-10-04   S08W91    10 Axx   1
12176 2014-10-06   N09W92    20 Cao   3
12181 2014-10-09   S14W88    50 Hsx   1
12178 2014-10-10   S01W93    60 Hax   1
12187 2014-10-25   S08W93    70 Hax   1
12192 2014-10-31   S15W94  

In [18]:
m["side"]    = m["location"].str[3]
m["cm_dist"] = m["location"].str[4:6].astype(int)

un = m[m["corr_UP"].isna()]
print(un["side"].value_counts())
n_limb = (un["cm_dist"] >= 77).sum()
print(n_limb, "of", len(un), "unmatched are 77 degrees or more from the central meridian")

side
W    24
Name: count, dtype: int64
20 of 24 unmatched are 77 degrees or more from the central meridian


In [19]:
R_MAX = 0.8
kept = m[m["corr_UP"].notna() & (m["area"] > 0) & (m["r"] < R_MAX)].copy()
print(len(kept), "rows kept of", len(m))
print(kept["ratio"].describe().round(2).to_string())
assert (kept["r"] < R_MAX).all() and (kept["area"] > 0).all()

for rmax in [0.6, 0.8, 0.9]:
    s = m[m["corr_UP"].notna() & (m["area"] > 0) & (m["r"] < rmax)]
    print(rmax, len(s), "rows | median ratio", round(s["ratio"].median(), 3))

345 rows kept of 544
count    345.00
mean       0.86
std        0.95
min        0.12
25%        0.50
50%        0.65
75%        0.81
max        9.90
0.6 241 rows | median ratio 0.669
0.8 345 rows | median ratio 0.653
0.9 410 rows | median ratio 0.637


In [20]:
for rmax in [0.6, 0.8, 0.9, 0.95]:
    s = m[m["corr_UP"].notna() & (m["area"] > 0) & (m["r"] < rmax)]
    q1, q3 = s["ratio"].quantile([0.25, 0.75])
    print(rmax, len(s), "rows | median", round(s["ratio"].median(), 3),
          "| IQR", round(q3 - q1, 3))

0.6 241 rows | median 0.669 | IQR 0.359
0.8 345 rows | median 0.653 | IQR 0.307
0.9 410 rows | median 0.637 | IQR 0.288
0.95 449 rows | median 0.63 | IQR 0.301
